# Notebook 04: Gradient Boosting

## Why This Matters

Gradient boosting dominates tabular ML competitions (XGBoost, LightGBM, CatBoost win the vast majority of Kaggle tabular competitions) and is widely deployed in production for fraud detection, CTR prediction, pricing models, and risk scoring. Unlike random forests (parallel training), gradient boosting is sequential - each tree corrects the errors of the previous ensemble. Understanding the mathematical intuition (fitting pseudo-residuals), the key algorithmic innovations in XGBoost/LightGBM, and the hyperparameter landscape is essential for applied ML roles.

In [ ]:
%pip install -q xgboost lightgbm optuna scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Gradient Boosting: The Core Intuition

Gradient boosting builds an additive model: `F_m(x) = F_{m-1}(x) + eta * h_m(x)`

Where `h_m` is a new tree fitted to the **pseudo-residuals** (negative gradient of the loss).

**For MSE loss (regression)**: gradient = predicted - actual = residuals. Each new tree literally fits the residuals of the current ensemble.

**For log-loss (classification)**: gradient = predicted_probability - actual_label. Each tree corrects the probability calibration.

**Learning rate eta**: shrinks each tree's contribution. Small eta = more trees needed but better generalization. Typical: 0.05-0.3.

In [ ]:
class GradBoostScratch:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators; self.learning_rate = learning_rate; self.max_depth = max_depth
        self.trees_ = []; self.F0_ = None
    
    def fit(self, X, y):
        self.F0_ = y.mean(); F = np.full(len(y), self.F0_)
        for _ in range(self.n_estimators):
            residuals = y - F  # negative gradient of MSE
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self
    
    def predict(self, X):
        F = np.full(len(X), self.F0_)
        for tree in self.trees_: F += self.learning_rate * tree.predict(X)
        return F

from sklearn.metrics import mean_squared_error
X, y = make_regression(n_samples=1000, n_features=10, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
gbm = GradBoostScratch(n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_tr, y_tr)
from sklearn.ensemble import GradientBoostingRegressor
sklearn_gbm = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_tr, y_tr)
print(f"From-scratch GBM RMSE: {np.sqrt(mean_squared_error(y_te, gbm.predict(X_te))):.4f}")
print(f"sklearn GBM RMSE:      {np.sqrt(mean_squared_error(y_te, sklearn_gbm.predict(X_te))):.4f}")

## 2. XGBoost Innovations

XGBoost (Chen & Guestrin, 2016) adds critical improvements:
1. **Regularized objective**: L1 (alpha) + L2 (lambda) on leaf weights + gamma penalizing number of leaves
2. **Second-order gradients**: Newton's method (hessian) for better tree structure decisions
3. **Approximate split finding**: histogram approximation for large datasets
4. **Sparsity-aware**: built-in handling of missing values

**Key hyperparameters**:
- `n_estimators`: 300-3000 (use early stopping)
- `learning_rate`: 0.01-0.3 (lower needs more trees)
- `max_depth`: 3-10 (deeper = more complex interactions)
- `subsample`: 0.6-1.0 (reduce variance)
- `colsample_bytree`: 0.6-1.0 (reduce variance)
- `min_child_weight`: like min_samples_leaf (regularization)

In [ ]:
X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss",
    early_stopping_rounds=20, random_state=42, verbosity=0)
xgb_model.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)
print(f"XGBoost - Best iter: {xgb_model.best_iteration}, Test AUC: {roc_auc_score(y_te, xgb_model.predict_proba(X_te)[:,1]):.4f}")

## 3. LightGBM: Why It's Faster

LightGBM (Ke et al., 2017) is 10-20x faster through:

1. **Leaf-wise (best-first) growth**: grows the leaf with highest gain, not level-by-level. Finds more complex splits. Control complexity with `num_leaves` not `max_depth`.

2. **GOSS** (Gradient-based One-Side Sampling): keeps all high-gradient samples + random sample of low-gradient samples. Reduces data while preserving signal.

3. **EFB** (Exclusive Feature Bundling): bundles mutually exclusive sparse features. Reduces effective feature count.

In [ ]:
import time

lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20, is_unbalance=False,
    random_state=42, verbosity=-1)
cbs = [lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)]
lgb_model.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], callbacks=cbs)
print(f"LightGBM - Best iter: {lgb_model.best_iteration_}, Test AUC: {roc_auc_score(y_te, lgb_model.predict_proba(X_te)[:,1]):.4f}")

X_large, y_large = make_classification(n_samples=50000, n_features=50, n_informative=20, random_state=42)
t = time.time(); xgb.XGBClassifier(n_estimators=200, verbosity=0).fit(X_large, y_large); xgb_t = time.time()-t
t = time.time(); lgb.LGBMClassifier(n_estimators=200, verbosity=-1).fit(X_large, y_large); lgb_t = time.time()-t
print(f"\nSpeed (50K samples, 200 trees): XGBoost={xgb_t:.2f}s, LightGBM={lgb_t:.2f}s, Speedup={xgb_t/lgb_t:.1f}x")

## 4. Hyperparameter Tuning with Optuna

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state": 42, "verbosity": 0
    }
    model = xgb.XGBClassifier(**params)
    return cross_val_score(model, X_tr, y_tr, cv=3, scoring="roc_auc").mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=False)
print(f"Best CV AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
best = xgb.XGBClassifier(**study.best_params, verbosity=0).fit(X_tr, y_tr)
print(f"Test AUC: {roc_auc_score(y_te, best.predict_proba(X_te)[:,1]):.4f}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Gradient boosting | Fit each tree to pseudo-residuals (negative gradient of loss) |
| Learning rate | Small eta = more trees but better generalization; use early stopping |
| XGBoost innovations | Regularized objective, second-order gradients, sparsity-aware |
| LightGBM speed | Leaf-wise growth + GOSS + EFB = 10-20x faster on large data |
| Early stopping | Prevent overfitting without manually tuning n_estimators |
| Optuna TPE | Bayesian search; better than random for expensive hyperparameter search |

### Common Pitfalls
- Setting max_depth too high without regularization (subsample, colsample_bytree)
- Not using early stopping
- Not scaling features (GBMs don't need scaling, but features for meta-learners do)
- Tuning n_estimators manually instead of using early stopping
